# FedEx Logistics Performance Analysis

**Goal:** assess delivery timeliness, operational bottlenecks, and logistics cost signals in the SCMS delivery history data. The dataset describes line-item records; unless explicitly aggregated, findings below are record-level and should not be read as unique-shipment counts.

### Questions
1. Does delivery mode relate to on-time performance?
2. Do country, managing team, or INCO terms show differences in delivery delay?
3. Is PO-to-vendor timing associated with delivery delay where a PO date exists?
4. Are shipment weight and line-item insurance cost associated?

**Method:** preserve raw fields, convert date and numeric fields with invalid/special text marked missing for calculations, document exclusions, and compare groups with sample sizes. Delays are actual delivery date minus scheduled date: positive = late; zero or negative = on-time. This is descriptive analysis, not proof of causation.

In [ ]:
!pip -q install seaborn plotly statsmodels

In [ ]:
import io, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 70)
print('Libraries ready')

## 1. Load the CSV

In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))
df_raw = pd.read_csv(io.BytesIO(uploaded[file_name]), low_memory=False)
df = df_raw.copy()
print(f'Loaded {file_name}: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 2. First look and data dictionary

Check dimensions, sample records, types, summary statistics, and duplicate records before cleaning.

In [ ]:
display(df.head())
display(df.tail())
print('Shape:', df.shape, '| Exact duplicate rows:', df.duplicated().sum())
df.info()
display(df.describe(include='all').T)

In [ ]:
data_dictionary = pd.DataFrame({
 'Column': df.columns,
 'Non-null': df.notna().sum().values,
 'Missing': df.isna().sum().values,
 'Unique': df.nunique(dropna=True).values,
 'Data type': df.dtypes.astype(str).values
})
display(data_dictionary)

### Field meanings

| Field | Meaning |
|---|---|
| ID | Unique logistics record identifier |
| Project Code | Project or initiative tied to the shipment |
| PQ # | Price quotation reference or process marker |
| PO / SO # | Purchase or sales order reference |
| ASN/DN # | Advance shipment notice or delivery note reference |
| Country | Destination country |
| Managed By | Team responsible for the project |
| Fulfill Via | Fulfillment path, such as direct drop or distribution center |
| Vendor INCO Term | Vendor delivery agreement term |
| Shipment Mode | Transport mode (air, truck, air charter, ocean) |
| PQ First Sent to Client Date | Date quotation was first sent to client |
| PO Sent to Vendor Date | Date purchase order was sent to vendor |
| Scheduled Delivery Date | Planned delivery date |
| Delivered to Client Date | Actual delivery date |
| Delivery Recorded Date | Date delivery was entered in the system |
| Product Group | Broad product category |
| Sub Classification | Product subgroup |
| Vendor | Supplier name |
| Item Description | Detailed shipped-item description |
| Molecule/Test Type | Pharmaceutical molecule or test type |
| Brand | Product brand |
| Dosage | Dose strength where applicable |
| Dosage Form | Product form, such as tablet or test kit |
| Unit of Measure (Per Pack) | Units contained in each pack |
| Line Item Quantity | Number of units/packs recorded for the line item |
| Line Item Value | Total value of the line item |
| Pack Price | Price per pack |
| Unit Price | Price per individual unit |
| Manufacturing Site | Product manufacturing location |
| First Line Designation | Whether product is designated for first-line use |
| Weight (Kilograms) | Recorded shipment/line-item weight in kilograms |
| Freight Cost (USD) | Freight cost recorded in US dollars |
| Line Item Insurance (USD) | Insurance cost recorded for the line item |

Some fields contain process labels or explanatory text instead of dates/numbers; those values remain in the raw data and are excluded from calculations requiring valid dates or numbers.

## 3. Clean and prepare analysis fields

The source includes text markers such as “Date Not Captured”, “N/A - From RDC”, “Weight Captured Separately”, and freight notes. These are retained in the raw table and become missing only in derived numeric/date analysis fields. We do not impute unknown dates, weight, freight, or insurance.

In [ ]:
date_cols = ['PQ First Sent to Client Date','PO Sent to Vendor Date','Scheduled Delivery Date','Delivered to Client Date','Delivery Recorded Date']
for c in date_cols:
    # mixed date formats occur in the source; placeholders safely become NaT
    df[c + ' Parsed'] = pd.to_datetime(df[c], format='mixed', errors='coerce')

numeric_cols = ['Unit of Measure (Per Pack)','Line Item Quantity','Line Item Value','Pack Price','Unit Price','Weight (Kilograms)','Freight Cost (USD)','Line Item Insurance (USD)']
for c in numeric_cols:
    if c == 'Unit of Measure (Per Pack)' or c in ['Line Item Quantity','Line Item Value','Pack Price','Unit Price','Line Item Insurance (USD)']:
        df[c + ' Numeric'] = pd.to_numeric(df[c].astype(str).str.replace(',','',regex=False).str.replace('$','',regex=False).str.strip(), errors='coerce')
    else:
        cleaned = df[c].astype(str).str.replace(',','',regex=False).str.replace('$','',regex=False).str.strip()
        df[c + ' Numeric'] = pd.to_numeric(cleaned, errors='coerce')

df['Delivery Delay Days'] = (df['Delivered to Client Date Parsed'] - df['Scheduled Delivery Date Parsed']).dt.days
df['On Time'] = df['Delivery Delay Days'].le(0)
df['Delivery Status'] = np.select([df['Delivery Delay Days'].isna(), df['Delivery Delay Days'] > 0], ['Unknown date','Late'], default='On time / early')
df['PO-to-Scheduled Lead Days'] = (df['Scheduled Delivery Date Parsed'] - df['PO Sent to Vendor Date Parsed']).dt.days
print('Unparseable / missing derived values:')
display(pd.DataFrame({'Missing after parse':df[[c+' Parsed' for c in date_cols]+[c+' Numeric' for c in numeric_cols]].isna().sum()}))
print('Delivery status:')
display(df['Delivery Status'].value_counts(dropna=False).to_frame('Records'))

## 4. Missing values and unusual values

Missing values are reported before decisions. Special text markers are not treated as zero. Outliers are inspected using distributions and IQR fences; valid extremes are retained unless source evidence shows an error, since unusually large shipments can be operationally real.

In [ ]:
missing = pd.DataFrame({'Missing':df_raw.isna().sum(), 'Missing %':(df_raw.isna().mean()*100).round(2)})
display(missing[missing['Missing']>0].sort_values('Missing',ascending=False))
for c in ['Delivery Delay Days','Weight (Kilograms) Numeric','Freight Cost (USD) Numeric','Line Item Insurance (USD) Numeric']:
    x=df[c].dropna()
    if len(x):
        q1,q3=x.quantile([.25,.75]); iqr=q3-q1
        print(f'{c}: n={len(x):,}, median={x.median():,.2f}, IQR=[{q1:,.2f}, {q3:,.2f}], IQR-fence count={((x<q1-1.5*iqr)|(x>q3+1.5*iqr)).sum():,}')

## 5. Delivery performance overview

In [ ]:
valid = df.dropna(subset=['Delivery Delay Days']).copy()
summary = pd.Series({
 'Records with usable scheduled and delivered dates':len(valid),
 'Late records':int((valid['Delivery Delay Days']>0).sum()),
 'On-time / early records':int((valid['Delivery Delay Days']<=0).sum()),
 'Late rate (%)':round((valid['Delivery Delay Days']>0).mean()*100,2),
 'On-time / early rate (%)':round((valid['Delivery Delay Days']<=0).mean()*100,2),
 'Median delay (days)':round(valid['Delivery Delay Days'].median(),2),
 'Mean delay (days)':round(valid['Delivery Delay Days'].mean(),2)
})
display(summary.to_frame('Value'))
print('Interpretation: the late rate uses only records with both scheduled and delivered dates; early/on-time is delay <= 0.')

## 6. Visual exploration (at least five chart types)

### Chart 1 — Histogram: delivery delay distribution

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(valid['Delivery Delay Days'], bins=50, kde=True, color='#2878B5')
plt.axvline(0,color='red',linestyle='--',label='Scheduled date (0 days)')
plt.xlim(valid['Delivery Delay Days'].quantile(.01),valid['Delivery Delay Days'].quantile(.99))
plt.title('Delivery delay distribution (central 98%)'); plt.xlabel('Days: positive = late, negative = early'); plt.legend(); plt.tight_layout(); plt.show()

### Chart 2 — Bar chart: on-time rate by shipment mode

Small groups are shown with sample sizes; missing mode remains visible.

In [ ]:
mode = df.dropna(subset=['Delivery Delay Days']).groupby('Shipment Mode',dropna=False).agg(records=('On Time','size'), on_time_rate=('On Time','mean'), median_delay=('Delivery Delay Days','median')).reset_index()
mode['On-time / early %']=mode['on_time_rate']*100
mode=mode[mode.records>=30].sort_values('On-time / early %')
plt.figure(figsize=(9,5)); sns.barplot(data=mode,x='On-time / early %',y='Shipment Mode',color='#4C9F70')
for i,(_,r) in enumerate(mode.iterrows()): plt.text(r['On-time / early %']+0.5,i,f"n={int(r['records']):,}",va='center',fontsize=9)
plt.xlim(0,105); plt.title('On-time / early delivery rate by shipment mode'); plt.tight_layout(); plt.show(); display(mode[['Shipment Mode','records','On-time / early %','median_delay']])

### Chart 3 — Box plot: delay by shipment mode

In [ ]:
plotdf=valid[valid['Shipment Mode'].notna()].copy()
order=plotdf.groupby('Shipment Mode')['Delivery Delay Days'].median().sort_values().index
plt.figure(figsize=(10,6)); sns.boxplot(data=plotdf,x='Delivery Delay Days',y='Shipment Mode',order=order,showfliers=False)
plt.xlim(plotdf['Delivery Delay Days'].quantile(.02),plotdf['Delivery Delay Days'].quantile(.98))
plt.title('Delivery delay by shipment mode (fliers hidden for readability)'); plt.xlabel('Days late / early'); plt.tight_layout(); plt.show()

### Chart 4 — Country comparison

Only countries with at least 30 date-valid records are compared to reduce unstable rates.

In [ ]:
country=valid.groupby('Country').agg(records=('On Time','size'),on_time_rate=('On Time','mean'),median_delay=('Delivery Delay Days','median')).query('records >= 30').sort_values('on_time_rate').tail(20).reset_index()
country['On-time / early %']=country.on_time_rate*100
fig=px.bar(country.sort_values('On-time / early %'),x='On-time / early %',y='Country',orientation='h',color='median_delay',color_continuous_scale='RdYlGn_r',hover_data=['records','median_delay'],title='On-time / early rate by country (minimum 30 records)')
fig.show()

### Chart 5 — Scatter plot: weight and insurance

This compares line-item values where both fields are numeric. Correlation is descriptive and can be influenced by product type, value, and shipment structure.

In [ ]:
wi=df.dropna(subset=['Weight (Kilograms) Numeric','Line Item Insurance (USD) Numeric']).copy()
wi=wi[(wi['Weight (Kilograms) Numeric']>0)&(wi['Line Item Insurance (USD) Numeric']>=0)]
print('Usable weight-insurance records:',len(wi))
if len(wi)>=3:
    print('Spearman correlation:',round(wi['Weight (Kilograms) Numeric'].corr(wi['Line Item Insurance (USD) Numeric'],method='spearman'),3))
    fig=px.scatter(wi,x='Weight (Kilograms) Numeric',y='Line Item Insurance (USD) Numeric',opacity=.45,trendline='ols',title='Weight vs line-item insurance (central 99% shown)',labels={'Weight (Kilograms) Numeric':'Weight (kg)','Line Item Insurance (USD) Numeric':'Insurance (USD)'})
    fig.update_xaxes(range=[0,wi['Weight (Kilograms) Numeric'].quantile(.99)]); fig.update_yaxes(range=[0,wi['Line Item Insurance (USD) Numeric'].quantile(.99)]); fig.show()

### Chart 6 — Correlation heatmap

In [ ]:
corr_cols=['Line Item Quantity Numeric','Line Item Value Numeric','Pack Price Numeric','Unit Price Numeric','Weight (Kilograms) Numeric','Freight Cost (USD) Numeric','Line Item Insurance (USD) Numeric','Delivery Delay Days']
corr=df[corr_cols].corr(method='spearman',numeric_only=True)
plt.figure(figsize=(10,8)); sns.heatmap(corr,annot=True,fmt='.2f',cmap='vlag',center=0,square=True)
plt.title('Spearman correlations among numeric measures'); plt.tight_layout(); plt.show()

### Chart 7 — Monthly line chart: median delay over time

In [ ]:
monthly=valid.dropna(subset=['Delivered to Client Date Parsed']).copy()
monthly['Delivery Month']=monthly['Delivered to Client Date Parsed'].dt.to_period('M').dt.to_timestamp()
monthly=monthly.groupby('Delivery Month').agg(records=('Delivery Delay Days','size'),median_delay=('Delivery Delay Days','median')).reset_index()
fig=px.line(monthly,x='Delivery Month',y='median_delay',hover_data=['records'],markers=True,title='Monthly median delivery delay',labels={'median_delay':'Median delay (days)'})
fig.add_hline(y=0,line_dash='dash',line_color='red'); fig.show()

### Chart 8 — INCO term performance (interactive)

INCO terms are compared as recorded. “N/A - From RDC” is a fulfillment category and may not be directly comparable to negotiated vendor terms.

In [ ]:
incoterm=valid.groupby('Vendor INCO Term').agg(records=('On Time','size'),on_time_rate=('On Time','mean'),median_delay=('Delivery Delay Days','median')).reset_index()
incoterm['On-time / early %']=incoterm.on_time_rate*100
incoterm=incoterm[incoterm.records>=30]
fig=px.bar(incoterm.sort_values('On-time / early %'),x='Vendor INCO Term',y='On-time / early %',color='median_delay',hover_data=['records','median_delay'],title='Delivery performance by INCO term (minimum 30 records)')
fig.show(); display(incoterm.sort_values('on_time_rate'))

## 7. Project milestone analyses

### PO-to-scheduled lead time vs delivery delay

The “N/A - From RDC” and uncaptured PO dates cannot provide a PO-to-scheduled interval; those records are excluded only from this question.

In [ ]:
lead=df.dropna(subset=['PO-to-Scheduled Lead Days','Delivery Delay Days']).copy()
lead=lead[(lead['PO-to-Scheduled Lead Days']>=0)&(lead['PO-to-Scheduled Lead Days']<=3650)]
print('Records with usable non-negative lead time:',len(lead))
if len(lead)>=3:
    print('Spearman correlation with delay:',round(lead['PO-to-Scheduled Lead Days'].corr(lead['Delivery Delay Days'],method='spearman'),3))
    lead['Lead-time band']=pd.cut(lead['PO-to-Scheduled Lead Days'],bins=[-1,30,60,90,180,365,3650],labels=['0–30','31–60','61–90','91–180','181–365','>365'])
    lead_summary=lead.groupby('Lead-time band',observed=True).agg(records=('On Time','size'),late_rate=('Delivery Delay Days',lambda x:(x>0).mean()*100),median_delay=('Delivery Delay Days','median')).reset_index()
    display(lead_summary)
    fig=px.bar(lead_summary,x='Lead-time band',y='late_rate',hover_data=['records','median_delay'],title='Late rate by PO-to-scheduled lead-time band',labels={'late_rate':'Late records (%)'})
    fig.show()

## 8. Findings and stakeholder actions

The following summary is generated from the loaded file; verify group sample sizes before acting. Treat differences as signals for further investigation, not causal effects.

In [ ]:
late_rate=(valid['Delivery Delay Days']>0).mean()*100 if len(valid) else np.nan
mode_action=mode.sort_values('On-time / early %') if 'mode' in globals() else pd.DataFrame()
country_action=valid.groupby('Country').agg(records=('On Time','size'),on_time=('On Time','mean'),median_delay=('Delivery Delay Days','median')).query('records >= 30').sort_values('on_time')
ins_corr=wi['Weight (Kilograms) Numeric'].corr(wi['Line Item Insurance (USD) Numeric'],method='spearman') if len(wi)>=3 else np.nan
print(f'Overall late rate among {len(valid):,} date-valid records: {late_rate:.1f}%')
if len(mode_action):
    print('Shipment mode lowest on-time/early rate (n>=30):',mode_action.iloc[0]['Shipment Mode'],f"({mode_action.iloc[0]['On-time / early %']:.1f}%, n={int(mode_action.iloc[0]['records']):,})")
if len(country_action):
    print('Lowest on-time/early country among groups with n>=30:',country_action.index[0],f"({country_action.iloc[0]['on_time']*100:.1f}%, n={int(country_action.iloc[0]['records']):,})")
print('Weight-insurance Spearman correlation:',round(ins_corr,3) if pd.notna(ins_corr) else 'not enough usable data')
print('Recommended follow-up: review recurring late lanes/modes with operations; validate vendor, country and schedule planning factors; investigate weight-insurance pricing with product/value controls before changing rates.')

## 9. Limitations and next steps

- Records are line items; one shipment may appear in multiple rows. Shipment-level results require a validated shipment key and aggregation rule.
- Date validity and completeness affect every delivery metric. Unknown delivery dates are excluded, not assumed on-time.
- Country, team, mode, and INCO term are observational attributes; results do not establish that a factor caused delay.
- Weight, freight, and insurance include non-numeric source notes. Cost comparisons use only numeric entries and may have selection bias.
- Managing team is highly imbalanced in the source; avoid ranking teams with tiny sample sizes.
- Next: validate outlying records with source owners; compare modes on matched routes/product groups; model delay with route, time, vendor, and product controls; quantify cost per unit and freight share after confirming units and shipment aggregation.